# 08 — Recall-First Target-Aware DifIISR v3

This is a new ablation, not a replacement for Notebook 06.

**Primary goal:** improve frozen-YOLO GT-matched recall for targets whose bounding-box width **and** height are ≤24 px in the bicubic ×4 pre-encoder frame.

Paired methods:
- **BASE** — frozen DifIISR, no new guidance.
- **V3-A** — bounded pooled-SCR guidance.
- **V3-B** — bounded pooled-SCR + frozen neighbour-density reliability.

The runner is crash-safe and deterministic per image. Completed outputs, per-image diagnostics, logs and YOLO evaluation JSONs are persisted on Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=False)
from pathlib import Path
import os,sys,subprocess,json,re,hashlib,time
import numpy as np,pandas as pd,torch
import torch.nn.functional as F
from PIL import Image

PROJECT_REPO='https://github.com/shelly-serafimovich/target-aware-infrared-sr.git'
DIFIISR_REPO='https://github.com/zirui0625/DifIISR.git'
DIFIISR_COMMIT='09ca97ea48d481656dd8090e84099c963059ac41'
SEED=12345

DATASET_ROOT=Path('/content/drive/MyDrive/thermal-drone-dataset')
LR_DIR=DATASET_ROOT/'difiisr_baseline'/'valid'/'LR_difiisr_train_degradation_v2'
YOLO_WEIGHTS=DATASET_ROOT/'detector'/'weights'/'thermal_drone_yolo11n_corrected_v1_best.pt'
EXPERIMENT_ROOT=DATASET_ROOT/'target_aware_difiisr'/'v3_recall_pilot_v1'
EXPERIMENT_ROOT.mkdir(parents=True,exist_ok=True)
project_dir=Path('/content/target-aware-infrared-sr'); difiisr_dir=Path('/content/DifIISR')

def run(cmd,cwd=None):
    print('$',' '.join(map(str,cmd)),flush=True); subprocess.run(list(map(str,cmd)),cwd=cwd,check=True)

if not project_dir.exists(): run(['git','clone',PROJECT_REPO,project_dir])
else:
    run(['git','fetch','origin'],cwd=project_dir); run(['git','checkout','main'],cwd=project_dir); run(['git','pull','--ff-only','origin','main'],cwd=project_dir)
if not difiisr_dir.exists(): run(['git','clone',DIFIISR_REPO,difiisr_dir])
run(['git','fetch','--all'],cwd=difiisr_dir); run(['git','checkout','--detach',DIFIISR_COMMIT],cwd=difiisr_dir)

ANN_DIR=project_dir/'annotations'/'corrected_yolo'/'valid'/'labels'
ENV=Path('/content/difiisr-env'); PY=str(ENV/'bin'/'python')
if not Path(PY).exists():
    run([sys.executable,'-m','pip','install','-q','uv'])
    run([sys.executable,'-m','uv','venv','--python','3.10',ENV])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'torch==2.1.1','torchvision==0.16.1','xformers==0.0.23','numpy==1.25.2','scipy==1.9.3','packaging==24.2','setuptools==69.5.1'])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'basicsr==1.4.2','opencv-python','matplotlib','timm','pandas','pillow','scikit-learn','scikit-image','lpips','loguru','omegaconf','six','tqdm','albumentations','einops','imageio','gdown'])
run([sys.executable,'-m','pip','install','-q','ultralytics'])

weights=difiisr_dir/'weights'; weights.mkdir(exist_ok=True)
if not (weights/'DifIISR.pth').exists(): run([PY,'-m','gdown','1PhRvk1Dlp3CCrPkrxRfNbNZ3fNgviDVZ','-O',weights/'DifIISR.pth'])
if not (weights/'autoencoder_vq_f4.pth').exists(): run([PY,'-m','gdown','https://github.com/zsyOAOA/ResShift/releases/download/v2.0/autoencoder_vq_f4.pth','-O',weights/'autoencoder_vq_f4.pth'])

assert LR_DIR.exists() and ANN_DIR.exists() and YOLO_WEIGHTS.exists()
print('✓ setup ready | GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')


In [ ]:
# ---------- Recall-first pilot configuration ----------
PILOT_N_TARGET_IMAGES=200
PILOT_N_BACKGROUND_IMAGES=80
CORE=24; OUTER=48; TOP_K=15
SIGMA_FLOOR=0.013458
BLOB_K=1.5; SOFT_T=0.25; SOFT_MIN_SUPPORT=3.0
SCR_GATE=1.8; SCR_GATE_TEMP=0.25
ND_GATE=0.40; ND_GATE_TEMP=0.07; ND_FLOOR_WEIGHT=0.50
MAX_CANDIDATES=48; PROPOSAL_NMS=7
SCR_GOAL=2.5; GOAL_TEMP=0.25
GUIDANCE_STEPS=2; GUIDANCE_LR=0.02
LAMBDA_FID=10.0; LAMBDA_BG=20.0

CONFIG=dict(seed=SEED,pilot_n_target_images=PILOT_N_TARGET_IMAGES,pilot_n_background_images=PILOT_N_BACKGROUND_IMAGES,
    small_target_definition='upsampled pre-encoder bbox width<=24 AND height<=24',
    core=CORE,outer=OUTER,top_k=TOP_K,sigma_floor=SIGMA_FLOOR,blob_k=BLOB_K,soft_t=SOFT_T,
    soft_min_support=SOFT_MIN_SUPPORT,scr_gate=SCR_GATE,scr_gate_temp=SCR_GATE_TEMP,
    nd_gate=ND_GATE,nd_gate_temp=ND_GATE_TEMP,nd_floor_weight=ND_FLOOR_WEIGHT,
    max_candidates=MAX_CANDIDATES,proposal_nms=PROPOSAL_NMS,scr_goal=SCR_GOAL,goal_temp=GOAL_TEMP,
    guidance_steps=GUIDANCE_STEPS,guidance_lr=GUIDANCE_LR,lambda_fid=LAMBDA_FID,lambda_bg=LAMBDA_BG,
    difiisr_commit=DIFIISR_COMMIT,primary_selection_metric='GT-matched recall @ IoU>=0.25 on <=24x24 targets')
RUN_ID='cfg_'+hashlib.sha1(json.dumps(CONFIG,sort_keys=True,separators=(',',':')).encode()).hexdigest()[:10]
RUN_ROOT=EXPERIMENT_ROOT/'runs'/RUN_ID; RUN_ROOT.mkdir(parents=True,exist_ok=True)
(RUN_ROOT/'config.json').write_text(json.dumps(CONFIG,indent=2))
print('RUN_ID:',RUN_ID); print('RUN_ROOT:',RUN_ROOT)
print('Changing any hyperparameter creates a new run folder, so results cannot mix.')


In [ ]:
# Build/reuse deterministic pilot manifest.
MANIFEST_PATH=EXPERIMENT_ROOT/f'pilot_manifest_{PILOT_N_TARGET_IMAGES}t_{PILOT_N_BACKGROUND_IMAGES}b_seed{SEED}.csv'
_EXTS={'.png','.jpg','.jpeg'}; images=sorted(p for p in LR_DIR.iterdir() if p.suffix.lower() in _EXTS)
labels=sorted(ANN_DIR.glob('*.txt')); _RE=re.compile(r'^(.*?)_(?:jpg|png)\.rf\.')
def canon(name):
    s=Path(name).stem; m=_RE.match(s); return m.group(1) if m else s
imgs_by={}; labs_by={}
for p in images: imgs_by.setdefault(canon(p.name),[]).append(p)
for p in labels: labs_by.setdefault(canon(p.name),[]).append(p)
for d in (imgs_by,labs_by):
    for k in d: d[k]=sorted(d[k])
label_for={}
for k,ips in imgs_by.items():
    lps=labs_by.get(k,[]); exact={p.stem:p for p in lps}; used=set(); miss=[]
    for ip in ips:
        if ip.stem in exact: label_for[ip.name]=exact[ip.stem]; used.add(exact[ip.stem])
        else: miss.append(ip)
    for ip,lp in zip(miss,[p for p in lps if p not in used]): label_for[ip.name]=lp
def targets(lp,W,H):
    if lp is None or not lp.exists() or lp.stat().st_size==0:return []
    out=[]
    for line in lp.read_text().splitlines():
        q=line.split()
        if len(q)<5:continue
        _,cx,cy,bw,bh=map(float,q[:5]); out.append((cx*W,cy*H,bw*W,bh*H))
    return out
rows=[]
for ip in images:
    with Image.open(ip) as im:w,h=im.size
    lp=label_for.get(ip.name); ts=targets(lp,w*4,h*4); small=[t for t in ts if t[2]<=CORE and t[3]<=CORE]
    rows.append(dict(image=ip.name,label_path=str(lp) if lp else '',lr_width=w,lr_height=h,up_width=w*4,up_height=h*4,
                     n_targets=len(ts),n_small_targets=len(small),is_background=len(ts)==0))
all_df=pd.DataFrame(rows)
print('Validation:',len(all_df),'| <=24x24 targets:',int(all_df.n_small_targets.sum()),'| BG:',int(all_df.is_background.sum()))
if MANIFEST_PATH.exists(): pilot=pd.read_csv(MANIFEST_PATH); print('✓ reusing',MANIFEST_PATH)
else:
    rng=np.random.default_rng(SEED); pos=all_df[all_df.n_small_targets>0]; bg=all_df[all_df.is_background]
    ps=pos.iloc[np.sort(rng.choice(len(pos),min(PILOT_N_TARGET_IMAGES,len(pos)),replace=False))]
    bs=bg.iloc[np.sort(rng.choice(len(bg),min(PILOT_N_BACKGROUND_IMAGES,len(bg)),replace=False))]
    pilot=pd.concat([ps,bs]).sample(frac=1,random_state=SEED).reset_index(drop=True)
    pilot['pilot_group']=np.where(pilot.n_small_targets>0,'small_target','background'); pilot.to_csv(MANIFEST_PATH,index=False)
    print('✓ created',MANIFEST_PATH)
display(pilot.pilot_group.value_counts().to_frame('images')); print('small targets in pilot:',int(pilot.n_small_targets.sum()))


In [ ]:
# Candidate-proposal coverage preflight. GT is diagnostic-only; the runner never sees it.
@torch.no_grad()
def propose(y_up,crop_h,crop_w):
    g=y_up.mean(1,keepdim=True).float(); k=49
    mu=F.avg_pool2d(g,k,1,k//2); mu2=F.avg_pool2d(g*g,k,1,k//2)
    z=(g-mu)/torch.sqrt((mu2-mu*mu).clamp_min(0.)+1e-6).clamp_min(SIGMA_FLOOR)
    mx=F.max_pool2d(z,PROPOSAL_NMS,1,PROPOSAL_NMS//2); valid=z>=mx-1e-7
    spatial=torch.zeros_like(valid); half=OUTER//2; spatial[...,half:crop_h-half,half:crop_w-half]=True
    score=z.masked_fill(~(valid&spatial),-1e9); n=min(MAX_CANDIDATES,int((score>-1e8).sum()))
    if n<=0:return []
    inds=torch.topk(score.flatten(),n).indices; W=score.shape[-1]
    return [(int(i//W),int(i%W)) for i in inds]
cov=[]; device='cuda' if torch.cuda.is_available() else 'cpu'
for r in pilot[pilot.pilot_group=='small_target'].itertuples(index=False):
    arr=np.asarray(Image.open(LR_DIR/r.image).convert('RGB'),dtype=np.float32)/255.; y=torch.from_numpy(arr).permute(2,0,1)[None].to(device); y=(y-.5)/.5
    h,w=y.shape[-2:]; y=F.pad(y,(0,(64-w%64)%64,0,(64-h%64)%64),mode='reflect'); cs=propose(F.interpolate(y,scale_factor=4,mode='bicubic'),h*4,w*4)
    for ti,(cx,cy,bw,bh) in enumerate(targets(Path(r.label_path),w*4,h*4)):
        if bw<=CORE and bh<=CORE:
            hit=any(abs(x-cx)<=bw/2 and abs(yc-cy)<=bh/2 for yc,x in cs); cov.append((r.image,ti,bw,bh,hit))
coverage=pd.DataFrame(cov,columns=['image','target_index','w','h','covered'])
coverage.to_csv(EXPERIMENT_ROOT/f'candidate_coverage_{PILOT_N_TARGET_IMAGES}t_seed{SEED}.csv',index=False)
rate=coverage.covered.mean(); print(f'Candidate coverage: {rate:.1%} ({coverage.covered.sum()}/{len(coverage)})')
print('✓' if rate>=.85 else '⚠ Increase candidate coverage before tuning guidance strength.')


In [ ]:
# Generate / resume BASE, V3-A and V3-B. Output streams live and is also persisted by the runner.
RUNNER=project_dir/'scripts'/'v3_recall_guidance_runner.py'
DIFIISR_CONFIG=difiisr_dir/'configs'/'DifIISR_test.yaml'
cmd=[PY,str(RUNNER),'--manifest',str(MANIFEST_PATH),'--lr',str(LR_DIR),'--out',str(RUN_ROOT),
     '--config',str(RUN_ROOT/'config.json'),'--difiisr-config',str(DIFIISR_CONFIG)]
proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout: print(line,end='',flush=True)
rc=proc.wait()
if rc: raise RuntimeError(f'Runner failed ({rc}). Completed images remain safe in {RUN_ROOT}.')
print('✓ generation complete')


In [ ]:
# Frozen-YOLO pilot evaluation. Also fully resume-safe.
EVAL_SCRIPT=project_dir/'scripts'/'v3_recall_yolo_eval.py'
cmd=[sys.executable,str(EVAL_SCRIPT),'--manifest',str(MANIFEST_PATH),'--run-root',str(RUN_ROOT),
     '--weights',str(YOLO_WEIGHTS),'--core',str(CORE)]
proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout: print(line,end='',flush=True)
rc=proc.wait()
if rc: raise RuntimeError(f'YOLO evaluation failed ({rc}); completed per-image JSONs are preserved.')


In [ ]:
EVAL_ROOT=RUN_ROOT/'yolo_pilot_eval'
summary=pd.read_csv(EVAL_ROOT/'pilot_summary.csv'); paired=pd.read_csv(EVAL_ROOT/'paired_saved_lost.csv')
print('PRIMARY — <=24x24 SMALL-TARGET RECALL')
display(summary.sort_values('small_target_recall_iou025',ascending=False))
print('PAIRED SAVED/LOST vs BASE'); display(paired)
q=EVAL_ROOT/'recall_by_small_target_quartile.csv'
if q.exists(): print('Recall inside small-target regime by shared size quartiles'); display(pd.read_csv(q))
print('Run artifacts:',RUN_ROOT)
print('Selection rule: maximize small-target recall first; use false alarms and distortion as secondary constraints.')


## Decision rule
Do not run the full 2,114-image validation automatically.

First compare V3-A and V3-B against paired BASE on:
1. small-target recall (primary),
2. saved minus lost targets,
3. mean GT confidence,
4. background false alarms (secondary safety signal).

If neither improves recall, change one axis at a time (e.g. SCR goal or guidance LR).  
Once a formulation is promising, freeze it and restore the full evaluation suite from Notebook 06.
